# Final Demo · Livestock weather-exposure decision support

**End-to-end proof of concept.** Uses *synthetic* TX/IA/WI station observations for quick verification. Change the input directory and configuration after downloading real NOAA data. **Do not report any sample scores as empirical findings.**

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts"))
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("LivestockWeatherNotebook").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
from livestock_weather.settings import load_settings
cfg = load_settings(ROOT / "config" / "demo.json")


## Research question

For a livestock company comparing potential operating locations, which candidate state has *lower historical extreme-weather exposure* under explicit heat, cold, and heavy-rain criteria, and when might seasonal preparations be needed?

**Pipeline:** NOAA-shaped CSV → PySpark ETL → annual/seasonal exposure metrics → configurable relative risk score → rankings and recommendations.

In [ ]:
from make_demo_data import create_demo
from livestock_weather.pipeline import run_pipeline
raw_dir = create_demo(ROOT / "data" / "demo_raw")
results = run_pipeline(
    spark=spark,
    raw_glob=str(raw_dir / "*.csv"),
    stations_path=raw_dir / "ghcnd-stations.txt",
    output_dir=ROOT / "data" / "demo_processed",
    cfg=cfg,
)
print("SYNTHETIC demo dataset — not NOAA findings")
print("Clean station-day rows:", results["daily"].count())
results["daily"].select("station_id", "state", "date", "tmax_c", "tmin_c", "prcp_mm").show(6, False)

## Historical extreme-weather patterns
Rates = event days / valid observed days. Station rates are averaged to states.

In [ ]:
results["annual"].orderBy("state", "year").show(30, False)
results["seasonal"].orderBy("state", "season").show(20, False)
# Explicit Spark SQL sample for the Big Data course.
results["daily"].createOrReplaceTempView("clean_daily")
spark.sql("""SELECT state, COUNT(DISTINCT station_id) AS observed_stations,
                    ROUND(AVG(tmax_c), 1) AS mean_observed_tmax_c
             FROM clean_daily GROUP BY state ORDER BY state""").show()

## Weather-exposure ranking and location comparison
This index describes the chosen conditions in the included states, **not livestock impacts**. Scores are sensitive to weights and compared states.

In [ ]:
from livestock_weather.scoring import compare_states
results["scoring"].select("state", "heat_score", "cold_score", "rain_score", "overall_score", "rank").show(60, False)
compare_states(results["scoring"], ["TX", "IA", "WI"]).select("state", "overall_score", "rank").show()

In [ ]:
from livestock_weather.visuals import plot_ranking
from IPython.display import Image, display
png = plot_ranking(results["scoring"], ROOT / "reports" / "figures" / "demo_state_ranking.png")
display(Image(filename=str(png)))

## Interpretation checklist for the final presentation

- Replace the **synthetic** demo with downloaded NOAA data and production settings.
- Which state has lower exposure **under the chosen weights**? Which hazard dominates its score?
- What are the seasonal exposure patterns, and how could a livestock operator prepare?
- Are all states supported by enough stations and years? Discuss nonuniform station location and missingness.
- Try changing risk weights / event thresholds and compare rankings.
- This is preliminary weather exposure assessment, **not** an optimized farm site or validated economic risk model.
- Add Spark input/output record counts, runtime, and Parquet/CSV comparison for Big Data course relevance.